## Load Dataset

Download the Amazon Product Reviews dataset from the provided Kaggle link (https://www.kaggle.com/datasets/bittlingmayer/amazonreviews) and load it into a pandas DataFrame. Focus on relevant columns like review text.


In [ ]:
import sys
!{sys.executable} -m pip install kaggle
print("Kaggle library installed successfully.")

Kaggle library installed successfully.


In [14]:
import os

# Create the .kaggle directory if it doesn't exist
!mkdir -p ~/.kaggle

# Move the kaggle.json file to the .kaggle directory
# Make sure to upload kaggle.json to the current working directory first
#!mv kaggle.json ~/.kaggle/

# Set permissions for the kaggle.json file
!chmod 600 ~/.kaggle/kaggle.json

print("Kaggle API credentials configured successfully.")

Kaggle API credentials configured successfully.


## Download Dataset


In [15]:
import os

# Download the dataset using the Kaggle API
!kaggle datasets download -d bittlingmayer/amazonreviews

# Unzip the downloaded file
# The dataset name is typically the last part of the Kaggle ID, so it should be amazonreviews.zip
!unzip -o amazonreviews.zip

print("Dataset downloaded and unzipped successfully.")

Dataset URL: https://www.kaggle.com/datasets/bittlingmayer/amazonreviews
License(s): unknown
 98% 482M/493M [00:06<00:00, 83.1MB/s]
100% 493M/493M [00:06<00:00, 75.9MB/s]
Archive:  amazonreviews.zip
  inflating: test.ft.txt.bz2         
  inflating: train.ft.txt.bz2        
Dataset downloaded and unzipped successfully.


In [17]:
import pandas as pd
import bz2

def load_fasttext_data(file_path):
    labels = []
    texts = []
    # Use bz2.open to handle .bz2 compressed files
    # 'rt' mode for reading text, encoding='utf-8' with errors='ignore' for robustness
    with bz2.open(file_path, 'rt', encoding='utf-8', errors='ignore') as f:
        for line in f:
            # FastText format: __label__X review text
            parts = line.strip().split(' ', 1)
            if len(parts) == 2:
                label = parts[0]
                text = parts[1]
                labels.append(label)
                texts.append(text)
    return pd.DataFrame({'label': labels, 'review_text': texts})

# Load the training data
df_train = load_fasttext_data('train.ft.txt.bz2')

# Load the testing data (though the main task focuses on reviews, it's good to load both if they are part of the dataset)
df_test = load_fasttext_data('test.ft.txt.bz2')

print("Training data loaded successfully. Displaying first 5 rows:")
print(df_train.head())
print("\nTesting data loaded successfully. Displaying first 5 rows:")
print(df_test.head())

print(f"\nShape of training data: {df_train.shape}")
print(f"Shape of testing data: {df_test.shape}")

Training data loaded successfully. Displaying first 5 rows:
        label                                        review_text
0  __label__2  Stuning even for the non-gamer: This sound tra...
1  __label__2  The best soundtrack ever to anything.: I'm rea...
2  __label__2  Amazing!: This soundtrack is my favorite music...
3  __label__2  Excellent Soundtrack: I truly like this soundt...
4  __label__2  Remember, Pull Your Jaw Off The Floor After He...

Testing data loaded successfully. Displaying first 5 rows:
        label                                        review_text
0  __label__2  Great CD: My lovely Pat has one of the GREAT v...
1  __label__2  One of the best game music soundtracks - for a...
2  __label__1  Batteries died within a year ...: I bought thi...
3  __label__2  works fine, but Maha Energy is better: Check o...
4  __label__2  Great for the non-audiophile: Reviewed quite a...

Shape of training data: (3600000, 2)
Shape of testing data: (400000, 2)


## Prepare the dataset for analysis


In [18]:
import pandas as pd

# 1. Concatenate the df_train and df_test DataFrames
df_reviews = pd.concat([df_train, df_test], ignore_index=True)

# 2. Randomly sample a subset of reviews
# Choosing 100,000 reviews for a manageable yet representative sample
sample_size = 100000
df_sample = df_reviews.sample(n=sample_size, random_state=42).reset_index(drop=True)

# 3. Create a new 'sentiment' column by mapping the 'label' column values
sentiment_mapping = {
    '__label__1': 'negative',
    '__label__2': 'positive'
}
df_sample['sentiment'] = df_sample['label'].map(sentiment_mapping)

# 4. Display the first few rows of df_sample and check its shape
print(f"Shape of the sampled DataFrame: {df_sample.shape}")
print("\nFirst 5 rows of the sampled DataFrame:")
print(df_sample.head())

Shape of the sampled DataFrame: (100000, 3)

First 5 rows of the sampled DataFrame:
        label                                        review_text sentiment
0  __label__1  Deeply disappointing, faulty morality & social...  negative
1  __label__2  insight into the philosophy of libertarian soc...  positive
2  __label__2  a great book: "In vain did the Bedouins strive...  positive
3  __label__1  toys for great sex: wow, that was bad, I threw...  negative
4  __label__2  i love this movie!!!!: i just finished reading...  positive


## Install spaCy and Download Model




In [19]:
import sys

# Install spaCy
!{sys.executable} -m pip install spacy

# Download the 'en_core_web_sm' English language model
!{sys.executable} -m spacy download en_core_web_sm

print("spaCy installed and 'en_core_web_sm' model downloaded successfully.")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 77.8 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.
spaCy installed and 'en_core_web_sm' model downloaded successfully.


## Perform Named Entity Recognition (NER)


In [20]:
import spacy
from tqdm.notebook import tqdm

# Load the English language model
nlp = spacy.load('en_core_web_sm')

# Extend the list of stop words to include common review-related terms that are not entities
# This is an optional step to refine noun chunk extraction by making them more meaningful
# Example: if 'great product' is a noun chunk, 'product' might be less informative alone
# than the actual name of the product.
# For now, let's stick to default spaCy entities and noun chunks, and filter for relevancy.

def extract_product_entities(text):
    doc = nlp(text)
    entities = set()

    # Extract named entities of types likely to be products or brands
    for ent in doc.ents:
        if ent.label_ in ('PRODUCT', 'ORG', 'WORK_OF_ART'):
            # Filter out very short or purely numeric entities that are unlikely to be product names
            if len(ent.text.strip()) > 1 and any(char.isalpha() for char in ent.text):
                entities.add(ent.text.strip())

    # Additionally, extract significant noun chunks as potential product names
    # Filter out noun chunks that are too generic or already covered by NER
    for chunk in doc.noun_chunks:
        chunk_text = chunk.text.strip()
        # Consider noun chunks that are not too short and contain at least one alphabet character
        # and are not common stop words or already found as a more specific entity
        if len(chunk_text) > 1 and any(char.isalpha() for char in chunk_text) and chunk_text.lower() not in nlp.Defaults.stop_words:
            is_overlapping_with_entity = False
            for ent in doc.ents:
                if str(chunk_text) == str(ent.text): # Check for exact match first
                    is_overlapping_with_entity = True
                    break
                if chunk.start_char >= ent.start_char and chunk.end_char <= ent.end_char: # chunk inside entity
                    is_overlapping_with_entity = True
                    break
                if ent.start_char >= chunk.start_char and ent.end_char <= chunk.end_char: # entity inside chunk
                    is_overlapping_with_entity = True
                    break

            if not is_overlapping_with_entity: # Only add if not an exact match or overlap with a recognized entity
                entities.add(chunk_text)

    return list(entities)

# Apply the function to the 'review_text' column with a progress bar
tqdm.pandas(desc="Extracting entities")
df_sample['extracted_entities'] = df_sample['review_text'].progress_apply(extract_product_entities)

# Display the first few rows of df_sample with the new column
print("DataFrame with extracted entities:")
print(df_sample.head())

Extracting entities:   0%|          | 0/100000 [00:00<?, ?it/s]

DataFrame with extracted entities:
        label                                        review_text sentiment  \
0  __label__1  Deeply disappointing, faulty morality & social...  negative   
1  __label__2  insight into the philosophy of libertarian soc...  positive   
2  __label__2  a great book: "In vain did the Bedouins strive...  positive   
3  __label__1  toys for great sex: wow, that was bad, I threw...  negative   
4  __label__2  i love this movie!!!!: i just finished reading...  positive   

                                  extracted_entities  
0  [(the color spectrum, red-orange-yellow...viol...  
1  [instruction, its richest diversity, nature,"T...  
2  [a crusader, a well written HF story, G.A. Hen...  
3                            [great sex, your money]  
4  [this movie, the plot, the ending, the movie, ...  


In [21]:
import sys

# Install TextBlob
!{sys.executable} -m pip install textblob
!{sys.executable} -m textblob.download_corpora

print("TextBlob installed and corpora downloaded successfully.")

[nltk_data] Downloading package brown to /root/nltk_data...
[nltk_data]   Unzipping corpora/brown.zip.
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data] Downloading package averaged_perceptron_tagger_eng to
[nltk_data]     /root/nltk_data...
[nltk_data]   Unzipping taggers/averaged_perceptron_tagger_eng.zip.
[nltk_data] Downloading package conll2000 to /root/nltk_data...
[nltk_data]   Unzipping corpora/conll2000.zip.
[nltk_data] Downloading package movie_reviews to /root/nltk_data...
[nltk_data]   Unzipping corpora/movie_reviews.zip.
Finished.
TextBlob installed and corpora downloaded successfully.


In [22]:
from textblob import TextBlob
from tqdm.notebook import tqdm

def get_textblob_sentiment(text):
    analysis = TextBlob(text)
    # Classify sentiment based on polarity
    if analysis.sentiment.polarity > 0:
        return 'positive'
    else:
        return 'negative' # Consider 0 or negative polarity as 'negative'

# Apply the sentiment analysis function to the 'review_text' column
tqdm.pandas(desc="Analyzing sentiment with TextBlob")
df_sample['textblob_sentiment'] = df_sample['review_text'].progress_apply(get_textblob_sentiment)

# Display the first few rows of df_sample with the new 'textblob_sentiment' column
print("DataFrame with TextBlob sentiments:")
print(df_sample[['review_text', 'sentiment', 'extracted_entities', 'textblob_sentiment']].head())

# Also print a value count to see the distribution of sentiments
print("\nTextBlob Sentiment Distribution:")
print(df_sample['textblob_sentiment'].value_counts())

Analyzing sentiment with TextBlob:   0%|          | 0/100000 [00:00<?, ?it/s]

DataFrame with TextBlob sentiments:
                                         review_text sentiment  \
0  Deeply disappointing, faulty morality & social...  negative   
1  insight into the philosophy of libertarian soc...  positive   
2  a great book: "In vain did the Bedouins strive...  positive   
3  toys for great sex: wow, that was bad, I threw...  negative   
4  i love this movie!!!!: i just finished reading...  positive   

                                  extracted_entities textblob_sentiment  
0  [(the color spectrum, red-orange-yellow...viol...           negative  
1  [instruction, its richest diversity, nature,"T...           positive  
2  [a crusader, a well written HF story, G.A. Hen...           positive  
3                            [great sex, your money]           positive  
4  [this movie, the plot, the ending, the movie, ...           positive  

TextBlob Sentiment Distribution:
textblob_sentiment
positive    75407
negative    24593
Name: count, dtype: int64


In [23]:
print("\nSample of Reviews with Extracted Entities and Sentiments:")
print(df_sample[['review_text', 'extracted_entities', 'sentiment', 'textblob_sentiment']].head(10))


Sample of Reviews with Extracted Entities and Sentiments:
                                         review_text  \
0  Deeply disappointing, faulty morality & social...   
1  insight into the philosophy of libertarian soc...   
2  a great book: "In vain did the Bedouins strive...   
3  toys for great sex: wow, that was bad, I threw...   
4  i love this movie!!!!: i just finished reading...   
5  Works....: I have had one of these for years. ...   
6  What can I say....: I have been an avid fan of...   
7  Pay attention to the other reviews!: After we ...   
8  This is not a review but a discussion: I chose...   
9  Sad: This product could not be used. Very poor...   

                                  extracted_entities sentiment  \
0  [(the color spectrum, red-orange-yellow...viol...  negative   
1  [instruction, its richest diversity, nature,"T...  positive   
2  [a crusader, a well written HF story, G.A. Hen...  positive   
3                            [great sex, your money]  negati

## Summary of Methods and Key Observations

### Named Entity Recognition (NER)

For Named Entity Recognition, the **spaCy** library with the `en_core_web_sm` English language model was utilized. The process involved:
1.  **Loading the spaCy model**: `nlp = spacy.load('en_core_web_sm')`.
2.  **Extracting Pre-defined Entities**: Identified entities labeled as 'PRODUCT', 'ORG' (organizations, often brands), and 'WORK_OF_ART' (e.g., books, movies, music titles) from the review text. Short or purely numeric entities were filtered out to enhance relevance.
3.  **Extracting Noun Chunks**: To capture potential product names not explicitly tagged by spaCy's standard NER, significant noun chunks were also extracted. These chunks were filtered to exclude short, generic terms or those overlapping with already identified named entities.

**Key Observations from NER:**
*   spaCy's `PRODUCT` and `ORG` labels are quite effective in identifying explicit product names and brands.
*   Including `WORK_OF_ART` helped in capturing titles of media products which are common in reviews.
*   Noun chunks proved useful for identifying more descriptive product references or components that might not be a formal named entity but are central to the review (e.g., "the color spectrum", "great sex").
*   There can be some noise or less relevant extractions (e.g., general concepts like "instruction" or "your money" appearing as entities), which could be further refined with custom rules or a more domain-specific NER model if needed.

### Rule-Based Sentiment Analysis

Rule-based sentiment analysis was performed using the **TextBlob** library, which leverages lexicons and patterns to assign a polarity score to text.
1.  **Installation and Corpora Download**: TextBlob and its necessary corpora (e.g., NLTK data for brown, punkt, wordnet) were installed.
2.  **Sentiment Calculation**: For each review, TextBlob's `sentiment.polarity` was calculated, which ranges from -1 (most negative) to +1 (most positive).
3.  **Sentiment Classification**: A simple rule was applied:
    *   `polarity > 0` was classified as 'positive'.
    *   `polarity <= 0` was classified as 'negative' (including neutral and negative sentiments).

**Key Observations from Sentiment Analysis:**
*   **TextBlob provides a quick and accessible way to get sentiment scores without training a model.**
*   The distribution shows a higher number of 'positive' reviews (75,407) compared to 'negative' reviews (24,593) based on TextBlob's classification for the sampled dataset. This might indicate a general tendency for Amazon reviews to be positive or a characteristic of TextBlob's lexicon for this type of text.
*   Comparing `textblob_sentiment` to the `sentiment` column (derived from `__label__1` and `__label__2`): While the sample output shows some agreement, there are instances where TextBlob classifies a review as 'positive' while the original label is 'negative' (e.g., review 3: "toys for great sex: wow, that was bad, I threw..."). This highlights that rule-based methods can sometimes differ from human-labeled datasets or more advanced machine learning models, especially for nuanced language or sarcasm. This discrepancy suggests that for higher accuracy, a fine-tuned model might be more appropriate than a generic rule-based approach.

Overall, both NER and sentiment analysis provided initial insights into the review content, successfully extracting potential product mentions and determining their emotional tone.

## Summary:

### Data Analysis Key Findings

*   **Dataset Preparation**: The Amazon Product Reviews dataset, comprising 3,600,000 training and 400,000 testing reviews, was successfully downloaded, extracted, and combined. A manageable sample of 100,000 reviews was selected, and their original labels (`__label__1`, `__label__2`) were mapped to 'negative' and 'positive' sentiments, respectively.
*   **Named Entity Recognition (NER)**:
    *   spaCy's `en_core_web_sm` model successfully extracted named entities (PRODUCT, ORG, WORK_OF_ART) and significant noun chunks from the review texts.
    *   This process effectively identified potential product names, brands, and descriptive product references within the reviews.
    *   Some less relevant extractions were observed, particularly from generic noun chunks.
*   **Rule-Based Sentiment Analysis**:
    *   TextBlob was used for rule-based sentiment analysis, classifying 75,407 of the 100,000 sampled reviews as 'positive' and 24,593 as 'negative' (based on polarity > 0 for positive, and $\leq$ 0 for negative).
    *   Discrepancies were noted between TextBlob's classifications and the original dataset's sentiment labels for some reviews, indicating that generic rule-based sentiment analysis may not fully capture the nuance of human-labeled data.

### Insights or Next Steps

*   To improve the precision of NER, consider refining the entity extraction process by applying custom rules or training a domain-specific NER model to minimize noise and focus on truly relevant product/brand mentions.
*   For more accurate sentiment analysis, particularly given the observed discrepancies with TextBlob, explore the use of more advanced machine learning or deep learning models that can be fine-tuned on the Amazon review dataset. This could better handle nuanced language, sarcasm, and domain-specific sentiment expressions.
